# 00 — Setup
Creates the catalog, schemas and platform tables. Safe to re-run.

**Prerequisite (once, by a metastore admin):** a Unity Catalog *storage credential* backed by the
Terraform `databricks_access_connector_id` output, and an *external location* covering
`lakehouse_root` and `landing_root`. See README step 6.

In [ ]:
%run ./_common

In [ ]:
if CATALOG != "spark_catalog":
    managed = f" MANAGED LOCATION '{LAKEHOUSE_ROOT}/uc'" if LAKEHOUSE_ROOT else ""
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}{managed}")

for schema in ("bronze", "silver", "gold"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {fq('silver', 'quarantine')} (
  source_table   STRING,
  batch_id       STRING,
  record_key     STRING,
  record_json    STRING,
  failed_rules   ARRAY<STRING>,
  quarantined_at TIMESTAMP
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {fq('silver', 'processed_batches')} (
  table_name   STRING,
  batch_id     STRING,
  processed_at TIMESTAMP
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {fq('gold', 'dq_results')} (
  run_ts     TIMESTAMP,
  as_of_date DATE,
  check_name STRING,
  status     STRING,
  detail     STRING
) USING DELTA
""")
print("setup complete")